> **Gabarito.** Abra só depois de tentar resolver a versão do aluno.

# Aula 8 – Clustering e escolha de K

**Objetivo:** agrupar solos espectralmente parecidos e escolher o número de grupos de forma objetiva.

## Preparação
Rode esta célula primeiro. Ela importa as funções das aulas anteriores.

In [ ]:
import sys
from pathlib import Path
# Encontra a pasta aulas/ (onde está curso_utils.py), rodando de aulas/ ou de aulas/solucoes/
_aqui = Path.cwd()
_aulas = _aqui if (_aqui / 'curso_utils.py').exists() else _aqui.parent
sys.path.insert(0, str(_aulas))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from curso_utils import preparar_nir, pca_svd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
meta, X, wl = preparar_nir(n=500)
Z = pca_svd(X, 6)[0]   # clustering nos 6 primeiros PCs
print(Z.shape)

## Exercício 1 – Inércia para vários K

In [ ]:
def kmeans_grid(Z, ks):
    return np.array([KMeans(int(k), n_init=20, random_state=42).fit(Z).inertia_ for k in ks])

ks = np.arange(2, 11)
inercias = kmeans_grid(Z, ks)
plt.plot(ks, inercias, 'o-'); plt.xlabel('K'); plt.ylabel('Inércia'); plt.show()

## Exercício 2 – Cotovelo de Shi (2021)
1. Normalize K e inércia para [0, 1].
2. Em cada ponto interno, calcule o ângulo entre o vetor para o ponto anterior e o vetor para o seguinte.
3. Escolha o K de **menor** ângulo.

In [ ]:
def cotovelo_shi(ks, inercias):
    ks, inercias = np.asarray(ks, float), np.asarray(inercias, float)
    k = (ks - ks.min()) / np.ptp(ks)
    i = (inercias - inercias.min()) / np.ptp(inercias)
    angulos = np.full(len(ks), np.nan)
    for j in range(1, len(ks) - 1):
        v1 = [k[j - 1] - k[j], i[j - 1] - i[j]]
        v2 = [k[j + 1] - k[j], i[j + 1] - i[j]]
        c = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
        angulos[j] = np.degrees(np.arccos(np.clip(c, -1, 1)))
    return int(ks[np.nanargmin(angulos)]), angulos

melhor_k, angulos = cotovelo_shi(ks, inercias)
print('K escolhido:', melhor_k)

## Exercício 3 – Visualizar os clusters

In [ ]:
rotulos = KMeans(melhor_k, n_init=30, random_state=42).fit_predict(Z)
plt.scatter(Z[:, 0], Z[:, 1], c=rotulos, cmap='tab10', s=10)
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title(f'K = {melhor_k}'); plt.show()
meta['cluster'] = rotulos
meta.groupby('cluster')[['Clay_gkg', 'Sand_gkg']].mean().round(0)

## Exercício 4 – Comparar com o silhouette

In [ ]:
sil = [silhouette_score(Z, KMeans(int(k), n_init=20, random_state=42).fit_predict(Z)) for k in ks]
plt.plot(ks, sil, 'o-'); plt.axvline(melhor_k, color='red', ls='--', label='Shi')
plt.xlabel('K'); plt.ylabel('Silhouette'); plt.legend(); plt.show()
print('K pelo silhouette:', int(ks[np.argmax(sil)]))

## Checkpoint

Os dois métodos escolheram o mesmo K? Por que podem divergir? Os clusters diferem em argila/areia média?